# OpenHSI Hikrobot Camera Implementations

:::{.callout-tip}

This module can be imported using `from openhsi.cameras import *`

:::

Wrapper class and example code for getting images from the OpenHSI.

:::{.callout-tip}

To use the camera, you will need some calibration files. You can also generate these files following this [guide](https://openhsi.github.io/openhsi/tutorial_calibrate.html) which uses the [`calibrate` module](https://openhsi.github.io/openhsi/calibrate.html).

:::

In [ ]:
#| hide

# documentation extraction for class methods
from nbdev.showdoc import *

# unit tests using test_eq(...)
from fastcore.test import *

# monkey patching class methods using @patch
from fastcore.foundation import *
from fastcore.foundation import patch

# bring forth **kwargs from an inherited class for documentation
from fastcore.meta import delegates

# external
import numpy as np
import ctypes
import matplotlib.pyplot as plt
import warnings
from tqdm import tqdm
from functools import partial

# internal
from openhsi.capture import OpenHSI
from openhsi.shared import SharedOpenHSI

## Hikrobot Camera

Used for OpenHSI cameras built with a Hikrobot machine vision detector (tested with the [MV-CA013-21UM](https://www.hikrobotics.com/en/machinevision/visionproduct?typeId=78&id=149), a 1280$\times$1024 monochrome USB3 Vision camera).

This camera requires the Hikrobot MVS SDK which can be downloaded from [https://www.hikrobotics.com/en/machinevision/service/download](https://www.hikrobotics.com/en/machinevision/service/download). The SDK ships its Python bindings (`MvCameraControl_class`) inside the `MvImport` folder of the SDK samples rather than as a pip package.

:::{.callout-note}

`HikrobotCameraBase` will look for `MvCameraControl_class` on `sys.path` first, then fall back to the default SDK sample locations (`/opt/MVS/Samples/64/Python/MvImport` on Linux and `C:/Program Files (x86)/MVS/Development/Samples/Python/MvImport` on Windows). If you installed the SDK somewhere else, set the `MVS_PYTHON_PATH` environment variable to your `MvImport` directory.

:::

:::{.callout-warning}

Use an unpacked pixel format (`Mono8`, `Mono10`, or `Mono12`) in your settings file. Packed formats (e.g. `Mono12Packed`) are not decoded by `get_img`.

:::

In [ ]:
#| export cameras

@delegates()
class HikrobotCameraBase():
    """Core functionality for Hikrobot machine vision cameras (tested with MV-CA013-21UM).

        Any keyword-value pair arguments must match those avaliable in the settings file. HikrobotCamera expects the ones listed below:

        - `binxy`: number of pixels to bin in (x,y) direction
        - `win_resolution`: size of area on detector to readout (height, width)
        - `win_offset`: offsets (y,x) from edge of detector for a selective readout
        - `exposure_ms`: the camera exposure time to use
        - `pixel_format`: format of pixels readout from sensor, i.e. Mono8, Mono10, Mono12 (packed formats are not supported)
    """
    def __init__(self, serial_num:str = None, **kwargs):
        """Initialise Camera"""
        # https://www.hikrobotics.com/en/machinevision/service/download
        super().__init__(**kwargs)

        import importlib, os, sys
        try:
            self.mvs = importlib.import_module("MvCameraControl_class")
        except ModuleNotFoundError:
            search_paths = [os.environ["MVS_PYTHON_PATH"]] if "MVS_PYTHON_PATH" in os.environ else []
            search_paths += ["/opt/MVS/Samples/64/Python/MvImport",
                             "/opt/MVS/Samples/aarch64/Python/MvImport",
                             "C:/Program Files (x86)/MVS/Development/Samples/Python/MvImport"]
            sys.path.extend(p for p in search_paths if os.path.isdir(p))
            self.mvs = importlib.import_module("MvCameraControl_class")

        # newer MVS SDK versions require explicit initialisation
        if hasattr(self.mvs.MvCamera, "MV_CC_Initialize"):
            self.mvs.MvCamera.MV_CC_Initialize()

        dev_list = self.mvs.MV_CC_DEVICE_INFO_LIST()
        self._check(self.mvs.MvCamera.MV_CC_EnumDevices(self.mvs.MV_GIGE_DEVICE | self.mvs.MV_USB_DEVICE, dev_list),
                    "enumerating devices")
        if dev_list.nDeviceNum == 0:
            raise RuntimeError("No Hikrobot camera found. Please connect a camera and run again.")

        dev_info = None
        for i in range(dev_list.nDeviceNum):
            info = ctypes.cast(dev_list.pDeviceInfo[i], ctypes.POINTER(self.mvs.MV_CC_DEVICE_INFO)).contents
            if serial_num is None or self._device_serial(info) == serial_num:
                dev_info = info
                break
        if dev_info is None:
            raise RuntimeError(f"No Hikrobot camera with serial number {serial_num} found.")

        self.hikcam = self.mvs.MvCamera()
        self._check(self.hikcam.MV_CC_CreateHandle(dev_info), "creating device handle")
        self._check(self.hikcam.MV_CC_OpenDevice(self.mvs.MV_ACCESS_Exclusive, 0), "opening device")
        print(f"Connected to device {self._device_serial(dev_info)}")

        # free-run acquisition with manual exposure/gain so capture matches the calibration data
        self._check(self.hikcam.MV_CC_SetEnumValue("TriggerMode", self.mvs.MV_TRIGGER_MODE_OFF), "turning TriggerMode off")
        self.hikcam.MV_CC_SetEnumValueByString("ExposureAuto", "Off")
        self.hikcam.MV_CC_SetEnumValueByString("GainAuto", "Off")
        self.set_gain(0) # default to 0 as we need to match to calibration data

        # binning is not available on all models so only warn if it cannot be set
        if self.hikcam.MV_CC_SetEnumValue("BinningHorizontal", self.settings["binxy"][0]) != 0 and self.settings["binxy"][0] > 1:
            warnings.warn(f"Could not set BinningHorizontal to {self.settings['binxy'][0]}.")
        if self.hikcam.MV_CC_SetEnumValue("BinningVertical", self.settings["binxy"][1]) != 0 and self.settings["binxy"][1] > 1:
            warnings.warn(f"Could not set BinningVertical to {self.settings['binxy'][1]}.")

        self._check(self.hikcam.MV_CC_SetEnumValueByString("PixelFormat", self.settings["pixel_format"]),
                    f"setting PixelFormat to {self.settings['pixel_format']} (check your camera supports it)")

        # always reset to no window.
        self.hikcam.MV_CC_SetIntValue("OffsetX", 0)
        self.hikcam.MV_CC_SetIntValue("OffsetY", 0)

        # set window up.
        self._check(self.hikcam.MV_CC_SetIntValue("Height", self.settings["win_resolution"][0] if self.settings["win_resolution"][0] > 0 else self._get_int("Height").nMax), "setting Height")
        self._check(self.hikcam.MV_CC_SetIntValue("Width",  self.settings["win_resolution"][1] if self.settings["win_resolution"][1] > 0 else self._get_int("Width").nMax),  "setting Width")
        self._check(self.hikcam.MV_CC_SetIntValue("OffsetY", self.settings["win_offset"][0] if self.settings["win_offset"][0] > 0 else 0), "setting OffsetY")
        self._check(self.hikcam.MV_CC_SetIntValue("OffsetX", self.settings["win_offset"][1] if self.settings["win_offset"][1] > 0 else 0), "setting OffsetX")

        self.set_exposure(self.settings["exposure_ms"])

        self.rows, self.cols = self._get_int("Height").nCurValue, self._get_int("Width").nCurValue

        # preallocate a reusable frame buffer
        self.payload_size = self._get_int("PayloadSize").nCurValue
        self.frame_buf = (ctypes.c_ubyte * self.payload_size)()
        self.frame_info = self.mvs.MV_FRAME_OUT_INFO_EX()

    def _check(self, ret:int, msg:str):
        if ret != 0:
            raise RuntimeError(f"MVS SDK error 0x{ret & 0xFFFFFFFF:08X} while {msg}.")

    def _get_int(self, name:str):
        val = self.mvs.MVCC_INTVALUE()
        self._check(self.hikcam.MV_CC_GetIntValue(name, val), f"getting {name}")
        return val

    def _device_serial(self, dev_info) -> str:
        if dev_info.nTLayerType == self.mvs.MV_USB_DEVICE:
            raw = bytes(dev_info.SpecialInfo.stUsb3VInfo.chSerialNumber)
        elif dev_info.nTLayerType == self.mvs.MV_GIGE_DEVICE:
            raw = bytes(dev_info.SpecialInfo.stGigEInfo.chSerialNumber)
        else:
            return ""
        return raw.split(b"\x00")[0].decode(errors="ignore")

    def __exit__(self, *args, **kwargs):
        self.hikcam.MV_CC_StopGrabbing()
        self.hikcam.MV_CC_CloseDevice()
        self.hikcam.MV_CC_DestroyHandle()

    def set_exposure(self, exposure_ms:float):
        self._check(self.hikcam.MV_CC_SetFloatValue("ExposureTime", exposure_ms*1000.0), "setting ExposureTime")
        val = self.mvs.MVCC_FLOATVALUE()
        self._check(self.hikcam.MV_CC_GetFloatValue("ExposureTime", val), "getting ExposureTime")
        self.settings["exposure_ms"] = val.fCurValue/1000.0 # exposure time rounds, so storing actual value

    def set_gain(self, gain_db:float):
        self.hikcam.MV_CC_SetFloatValue("Gain", gain_db*1.0)

    def start_cam(self):
        self._check(self.hikcam.MV_CC_StartGrabbing(), "starting acquisition")

    def stop_cam(self):
        self.hikcam.MV_CC_StopGrabbing()

    def get_img(self) -> np.ndarray:
        timeout_ms = int(2*self.settings["exposure_ms"]) + 1000
        self._check(self.hikcam.MV_CC_GetOneFrameTimeout(ctypes.byref(self.frame_buf), self.payload_size, self.frame_info, timeout_ms),
                    "grabbing frame")
        img = np.frombuffer(self.frame_buf, dtype=np.uint8, count=self.frame_info.nFrameLen)
        if self.frame_info.nFrameLen == 2*self.frame_info.nHeight*self.frame_info.nWidth:
            img = img.view(np.uint16) # Mono10/Mono12/Mono16 are delivered as 2 bytes per pixel
        elif self.frame_info.nFrameLen != self.frame_info.nHeight*self.frame_info.nWidth:
            raise RuntimeError("Unexpected frame length. Use an unpacked pixel format (Mono8, Mono10, or Mono12).")
        return img.reshape(self.frame_info.nHeight, self.frame_info.nWidth).copy()

    def get_temp(self) -> float:
        val = self.mvs.MVCC_FLOATVALUE()
        if self.hikcam.MV_CC_GetFloatValue("DeviceTemperature", val) == 0:
            return val.fCurValue
        return float("nan")

@delegates()
class HikrobotCamera(HikrobotCameraBase, OpenHSI):
    pass

In [ ]:
#| hardware

with HikrobotCamera(n_lines=128, exposure_ms=10, processing_lvl = -1,
                    cal_path="", json_path='../assets/cam_settings_hikrobot.json') as cam:
    cam.collect()

#fig = cam.show(robust=True)
#fig

### Multiprocessing camera export

Export cameras using the SharedOpenHSI class.

In [ ]:
#| export cameras

@delegates()
class SharedHikrobotCamera(HikrobotCameraBase, SharedOpenHSI):
    pass